# 🇧🇷 Lloavece-Coder: Treinamento Oficial na GPU A100 (Google Colab Pro)

Este notebook executa o treinamento industrial da **Lloavece-Coder** (LLM de Código com Alma Brasileira).
- **Hardware**: NVIDIA A100 (40GB/80GB VRAM) ou L4
- **Armazenamento**: Google Drive 5TB (Persistência contínua de checkpoints)
- **Precisão**: `bfloat16` nativo com FlashAttention (SDPA), RoPE e GQA

### 1. Conectar ao Google Drive de 5TB (Persistência Automática de Checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lloavece_checkpoints
!mkdir -p /content/drive/MyDrive/lloavece_data
print("[*] Google Drive de 5TB conectado e pastas inicializadas!")

### 2. Verificar a GPU de Alta Performance (A100 / L4)

In [ ]:
!nvidia-smi

### 3. Clonar ou Atualizar o Repositório Oficial Lloavece

In [ ]:
import os
# Suporte a repositório privado via Colab Secrets ou clone público
gh_token = None
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if not os.path.exists('Lloavece'):
    if gh_token:
        !git clone https://{gh_token}@github.com/Alekkzsx/Lloavece.git
    else:
        !git clone https://github.com/Alekkzsx/Lloavece.git
%cd Lloavece
!git pull origin main

!pip install sentencepiece datasets tqdm numpy crcmod -q
print("[*] Ambiente Lloavece configurado com sucesso!")

### 4. Coletar o Dataset Completo: Português Brasileiro (PT-BR) + Código

In [ ]:
# Coleta o currículo nacional (PIX, CLT, Boleto, LGPD, FastAPI) + 10.000 amostras PT-BR e código
!python -m src.data.collectors.download_code --samples 10000

# Treina o Tokenizer BPE Byte-Fallback de 32k com indentação e tokens FIM
!python -m src.tokenizer.code_tokenizer --vocab_size 32768 --samples 50000

# Compila em Memmap uint16 com 50% de taxa Fill-In-The-Middle (FIM)
!python -m src.data.pipeline --fim_rate 0.5

### 5. Iniciar o Pré-Treinamento na GPU A100 (Modelo 0.5B com RoPE + GQA)

In [ ]:
!python -m src.training.train \
    --config 0.5B \
    --precision bf16 \
    --batch_size 16 \
    --grad_accum_steps 8 \
    --max_iters 3000 \
    --lr 3e-4 \
    --eval_interval 100 \
    --drive_dir /content/drive/MyDrive/lloavece_checkpoints

### 6. Alinhamento e Fine-Tuning de Instruções Técnicas em PT-BR (SFT)

In [ ]:
!python -m src.training.sft \
    --checkpoint /content/drive/MyDrive/lloavece_checkpoints/best_model.pt \
    --drive_dir /content/drive/MyDrive/lloavece_checkpoints

### 7. Comprimir o Modelo com o Motor K-Bit Sparse (~0.15 bits/peso)

In [ ]:
!python -m src.export.kbit_compressor

### 8. Teste de Geração de Código em Português do Brasil

In [ ]:
!python -m src.generate \
    --checkpoint /content/drive/MyDrive/lloavece_checkpoints/best_model.pt \
    --prompt "def processar_pagamento_pix(chave: str, valor: float):"